# PageRank Algorithm — From Scratch

### Graph Details from Notes:
* **Nodes:** A, B, C, D ($N = 4$)
* **Outgoing Links:**
  * $A \rightarrow B, C$ ($L(A) = 2$)
  * $B \rightarrow C$ ($L(B) = 1$)
  * $C \rightarrow A$ ($L(C) = 1$)
  * $D \rightarrow C$ ($L(D) = 1$)
* **Damping Factor ($d$):** $0.85$
* **Teleportation Base Term:** $\frac{1 - d}{N} = \frac{1 - 0.85}{4} = 0.0375$

---

### PageRank Formula:
$$PR(P_i) = \frac{1 - d}{N} + d \sum_{P_j \in M(P_i)} \frac{PR(P_j)}{L(P_j)}$$

In [3]:
import numpy as np

## 1. Define the Graph and Parameters

We define the pages, outgoing links, and compute the base teleportation parameter $\frac{1-d}{N}$.

In [4]:
pages = ['A', 'B', 'C', 'D']
N = len(pages)
d = 0.85
base_term = (1 - d) / N  # (1 - 0.85) / 4 = 0.0375

links = {
    'A': ['B', 'C'],
    'B': ['C'],
    'C': ['A'],
    'D': ['C']
}

print("Pages:", pages)
print("Number of pages (N):", N)
print("Base teleportation term:", base_term)
print("Outgoing links:", links)

Pages: ['A', 'B', 'C', 'D']
Number of pages (N): 4
Base teleportation term: 0.037500000000000006
Outgoing links: {'A': ['B', 'C'], 'B': ['C'], 'C': ['A'], 'D': ['C']}


## 2. Construct the Transition Matrix

We create a column-stochastic matrix $M$, where column $j$ contains the transition probabilities from page $j$ to its destinations: $\frac{1}{L(j)}$.

In [5]:
M = np.zeros((N, N))

for j, page in enumerate(pages):
    outgoing = links[page]
    probability = 1.0 / len(outgoing)
    for destination in outgoing:
        i = pages.index(destination)
        M[i, j] = probability

print("Transition matrix M (rows: [A, B, C, D], cols: [A, B, C, D]):")
print(M)
print("\nColumn sums:")
print(M.sum(axis=0))

Transition matrix M (rows: [A, B, C, D], cols: [A, B, C, D]):
[[0.  0.  1.  0. ]
 [0.5 0.  0.  0. ]
 [0.5 1.  0.  1. ]
 [0.  0.  0.  0. ]]

Column sums:
[1. 1. 1. 1.]


## 3. Step 1: Initialization

Every page begins with equal initial rank[cite: 1]:
$$PR^{(0)}(P_i) = \frac{1}{N} = \frac{1}{4} = 0.25$$

In [6]:
rank = np.ones(N) / N

print("Step 1 (Initialization):")
for i in range(N):
    print(f"PR({pages[i]}) = {rank[i]:.4f}")

Step 1 (Initialization):
PR(A) = 0.2500
PR(B) = 0.2500
PR(C) = 0.2500
PR(D) = 0.2500


## 4. Step 2: First Iteration (Manual Verification)

Perform one iteration to verify against the handwritten notes:
* $PR(A) = 0.0375 + 0.85 \times \frac{0.25}{1} = 0.25$
* $PR(B) = 0.0375 + 0.85 \times \frac{0.25}{2} = 0.14375$
* $PR(C) = 0.0375 + 0.85 \times (\frac{0.25}{2} + \frac{0.25}{1} + \frac{0.25}{1}) = 0.56875$
* $PR(D) = 0.0375 + 0.85 \times 0 = 0.0375$

In [7]:
step2_rank = base_term + d * (M @ rank)

print("Step 2 (After 1st Iteration):")
for i in range(N):
    print(f"PR({pages[i]}) = {step2_rank[i]:.6f}")

Step 2 (After 1st Iteration):
PR(A) = 0.250000
PR(B) = 0.143750
PR(C) = 0.568750
PR(D) = 0.037500


## 5. Iteration Until Convergence

Apply the update rule iteratively until consecutive rank differences drop below the tolerance threshold:
$$\mathbf{r}_{new} = \frac{1-d}{N}\mathbf{1} + dM\mathbf{r}_{old}$$

In [8]:
max_iterations = 100
tolerance = 1e-10
current_rank = np.ones(N) / N

for iteration in range(1, max_iterations + 1):
    new_rank = base_term + d * (M @ current_rank)
    diff = np.sum(np.abs(new_rank - current_rank))
    current_rank = new_rank

    print(f"Iteration {iteration:2d}: {np.round(current_rank, 6)} | Diff = {diff:.2e}")

    if diff < tolerance:
        print(f"\nConverged at iteration {iteration}")
        break

Iteration  1: [0.25    0.14375 0.56875 0.0375 ] | Diff = 6.37e-01
Iteration  2: [0.520937 0.14375  0.297813 0.0375  ] | Diff = 5.42e-01
Iteration  3: [0.290641 0.258898 0.412961 0.0375  ] | Diff = 4.61e-01
Iteration  4: [0.388517 0.161022 0.412961 0.0375  ] | Diff = 1.96e-01
Iteration  5: [0.388517 0.20262  0.371364 0.0375  ] | Diff = 8.32e-02
Iteration  6: [0.353159 0.20262  0.406721 0.0375  ] | Diff = 7.07e-02
Iteration  7: [0.383213 0.187593 0.391694 0.0375  ] | Diff = 6.01e-02
Iteration  8: [0.37044  0.200366 0.391694 0.0375  ] | Diff = 2.55e-02
Iteration  9: [0.37044  0.194937 0.397123 0.0375  ] | Diff = 1.09e-02
Iteration 10: [0.375054 0.194937 0.392509 0.0375  ] | Diff = 9.23e-03
Iteration 11: [0.371132 0.196898 0.39447  0.0375  ] | Diff = 7.84e-03
Iteration 12: [0.372799 0.195231 0.39447  0.0375  ] | Diff = 3.33e-03
Iteration 13: [0.372799 0.19594  0.393761 0.0375  ] | Diff = 1.42e-03
Iteration 14: [0.372197 0.19594  0.394363 0.0375  ] | Diff = 1.20e-03
Iteration 15: [0.372709 

## 6. Final PageRank Scores and Rankings

Sort and display the steady-state importance scores from highest to lowest.

In [9]:
rank_list = []
for i in range(N):
    rank_list.append((pages[i], current_rank[i]))

ranking = sorted(rank_list, key=lambda x: x[1], reverse=True)

print("Final PageRank scores:")
for i in range(N):
    print(f"{pages[i]}: {current_rank[i]:.6f}")

print("\nPages ranked from highest to lowest:")
for pos in range(len(ranking)):
    page_name, score = ranking[pos]
    print(f"{pos + 1}. Page {page_name} -> {score:.6f}")

Final PageRank scores:
A: 0.372527
B: 0.195824
C: 0.394149
D: 0.037500

Pages ranked from highest to lowest:
1. Page C -> 0.394149
2. Page A -> 0.372527
3. Page B -> 0.195824
4. Page D -> 0.037500


## 7. Reusable PageRank Function

Encapsulate the full algorithm into a general-purpose Python function.

In [10]:
def pagerank(links_dict, damping=0.85, max_iter=100, tol=1e-10):
    node_list = list(links_dict.keys())
    n = len(node_list)

    trans_matrix = np.zeros((n, n))
    for col_idx in range(n):
        src = node_list[col_idx]
        targets = links_dict[src]
        if len(targets) == 0:
            trans_matrix[:, col_idx] = 1.0 / n
        else:
            prob = 1.0 / len(targets)
            for dest in targets:
                row_idx = node_list.index(dest)
                trans_matrix[row_idx, col_idx] = prob

    r = np.ones(n) / n
    teleport = (1.0 - damping) / n

    for it in range(1, max_iter + 1):
        r_new = teleport + damping * (trans_matrix @ r)
        if np.sum(np.abs(r_new - r)) < tol:
            result_dict = {}
            for i in range(n):
                result_dict[node_list[i]] = r_new[i]
            return result_dict, it
        r = r_new

    result_dict = {}
    for i in range(n):
        result_dict[node_list[i]] = r[i]
    return result_dict, max_iter

scores, iters = pagerank(links)

print("Function Output:")
for node in sorted(scores, key=scores.get, reverse=True):
    print(f"Page {node}: {scores[node]:.6f}")
print(f"Converged in {iters} iterations.")

Function Output:
Page C: 0.394149
Page A: 0.372527
Page B: 0.195824
Page D: 0.037500
Converged in 47 iterations.
